这里说频率是指一个机场航班出现的频率

## 加载数据

数据已经预处理过了

In [1]:
import pandas as pd


# 加载数据
pd.set_option('display.max_columns', None)  # 显示所有列

data = pd.read_csv('./pre_2023-2024_with_comp_test.csv', dtype={'aircraft': str})

# 查看前几行数据，确保加载成功
print(data.head())

print(data.info())

  flt_no  cap aircraft  legs  leg_no  duration  pax    a    b    c  \
0   7558   94      190     1       1      1.33   45  AAT  URC  NaN   
1   5248  162      322     1       1      2.10  143  AKA  HGH  NaN   
2   3524  169      738     1       1      4.52  154  AKU  CGO  NaN   
3   6352  167      320     1       1      4.28  167  AKU  CGO  NaN   
4   7516  166      320     1       1      1.33  181  AKU  URC  NaN   

    unit_price  competitor_price  year  month  day  weekday  hour  minute  \
0   481.555556       -119.744444  2024      7    1        0    12      30   
1   902.167832          0.000000  2024      7    1        0    10      30   
2  1540.357143        283.690476  2024      7    1        0    14      15   
3  1345.449102         88.782435  2024      7    1        0    21      30   
4   593.480663        -54.049640  2024      7    1        0    23      15   

  from   to  
0  AAT  URC  
1  AKA  HGH  
2  AKU  CGO  
3  AKU  CGO  
4  AKU  URC  
<class 'pandas.core.frame.DataFr

In [2]:
# # 随机抽取5万条数据
# data = data.sample(n=20000, random_state=42)

# 查看抽样后的数据信息
print(data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 282755 entries, 0 to 282754
Data columns (total 20 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   flt_no            282755 non-null  object 
 1   cap               282755 non-null  int64  
 2   aircraft          282755 non-null  object 
 3   legs              282755 non-null  int64  
 4   leg_no            282755 non-null  int64  
 5   duration          282727 non-null  float64
 6   pax               282755 non-null  int64  
 7   a                 282755 non-null  object 
 8   b                 282755 non-null  object 
 9   c                 112308 non-null  object 
 10  unit_price        282755 non-null  float64
 11  competitor_price  282755 non-null  float64
 12  year              282755 non-null  int64  
 13  month             282755 non-null  int64  
 14  day               282755 non-null  int64  
 15  weekday           282755 non-null  int64  
 16  hour              28

In [3]:
# 检查标准化后的统计信息
print("\n标准化后的统计信息：")
print(data['pax'].describe())


标准化后的统计信息：
count    282755.000000
mean        132.496755
std          51.136985
min          20.000000
25%          99.000000
50%         141.000000
75%         165.000000
max         339.000000
Name: pax, dtype: float64


## 编码分类变量

### 新增城市标签

In [4]:
import json
# 加载字典
with open('../../my/encoder/city_labels_航班频率加权图标签.json', 'r') as file:
    city_labels_loaded = json.load(file)

print("加载的字典：", city_labels_loaded)

加载的字典： {'AAT': 2, 'CKG': 0, 'PEK': 0, 'TCG': 2, 'URC': 2, 'XIY': 0, 'YIN': 2, 'ACF': 2, 'HMI': 2, 'SHF': 2, 'TLQ': 2, 'ACX': 1, 'CGO': 0, 'KMG': 0, 'AEB': 1, 'CAN': 0, 'HGH': 0, 'KWL': 0, 'TFU': 0, 'AKA': 1, 'HAK': 0, 'TSN': 0, 'UYN': 0, 'AKU': 2, 'FOC': 0, 'HTN': 2, 'AOG': 0, 'TNA': 0, 'AQG': 0, 'KWE': 0, 'NGB': 0, 'TAO': 0, 'XMN': 0, 'AVA': 0, 'LLB': 0, 'BAR': 1, 'HRB': 0, 'JMJ': 1, 'KHN': 0, 'LHW': 0, 'SJW': 0, 'SZX': 0, 'BAV': 0, 'HLD': 1, 'TGO': 1, 'XIL': 1, 'BFJ': 0, 'NNG': 0, 'BHY': 0, 'HFE': 0, 'JNG': 0, 'NAO': 2, 'WUH': 0, 'XUZ': 0, 'YIH': 0, 'BPE': 0, 'CSX': 0, 'BPL': 2, 'BPX': 0, 'BSD': 1, 'BZX': 0, 'LJG': 0, 'NKG': 0, 'CGQ': 0, 'DLC': 0, 'DOY': 1, 'HDG': 0, 'HET': 0, 'HSN': 0, 'HZG': 2, 'INC': 0, 'JIQ': 0, 'PKX': 0, 'PVG': 0, 'SHA': 0, 'SHE': 0, 'SQJ': 0, 'SYX': 0, 'TEN': 0, 'TYN': 0, 'WEF': 0, 'WUA': 1, 'YIC': 0, 'ZQZ': 0, 'CGD': 0, 'JHG': 0, 'CIF': 0, 'DLU': 1, 'ENH': 0, 'GOQ': 2, 'HNY': 0, 'HUZ': 0, 'JGS': 0, 'JJN': 0, 'JNZ': 1, 'KCA': 2, 'KHG': 2, 'KOW': 0, 'KRL': 2, 'L

In [5]:
# 使用 map 对 'a', 'b', 'c', 'from', 'to' 列进行标签化，新增对应的标签列
data['a_label'] = data['a'].map(city_labels_loaded)
data['b_label'] = data['b'].map(city_labels_loaded)
data['c_label'] = data['c'].map(city_labels_loaded)
data['from_label'] = data['from'].map(city_labels_loaded)
data['to_label'] = data['to'].map(city_labels_loaded)

### 新增城市二维嵌入

In [6]:
import json

# 加载字典
with open('../../my/encoder/城市嵌入编码_航班频率加权图.json', 'r') as file:
    city_embeddings = json.load(file)

print("加载的字典：", city_embeddings)


加载的字典： {'AAT': [1.3597848415374756, 0.6660159826278687], 'CKG': [-0.0919976532459259, 0.6568402051925659], 'PEK': [-0.2624581456184387, 0.18506205081939697], 'TCG': [1.3452502489089966, 0.9850115776062012], 'URC': [1.191598653793335, 1.0679560899734497], 'XIY': [-0.5777866840362549, 0.9855008721351624], 'YIN': [1.9014595746994019, 0.7400669455528259], 'ACF': [1.4711376428604126, 0.9144527316093445], 'HMI': [0.39429816603660583, 0.8812238574028015], 'SHF': [1.0217688083648682, 0.19476288557052612], 'TLQ': [0.6065776944160461, 0.4862461984157562], 'ACX': [0.7235668301582336, -0.6459048390388489], 'CGO': [0.25691214203834534, 0.8677778244018555], 'KMG': [-0.3540305197238922, -0.0116242291405797], 'AEB': [0.7109003663063049, -1.2106705904006958], 'CAN': [-0.09886128455400467, 0.5134572982788086], 'HGH': [-0.2080717533826828, 0.4809180498123169], 'KWL': [-0.9176110625267029, 0.5989621877670288], 'TFU': [0.17415915429592133, 0.24932153522968292], 'AKA': [-0.07954857498407364, -0.708072543144

In [7]:
import pandas as pd
import numpy as np

# 将 city_embeddings 转换为 DataFrame
embedding_df = pd.DataFrame.from_dict(city_embeddings, orient='index', columns=['embedding_1', 'embedding_2'])
embedding_df.index.name = 'city'

# 用 'a', 'b', 'c', 'from', 'to' 字段与 embedding_df 合并
data = data.merge(embedding_df, left_on='a', right_index=True, how='left')
data.rename(columns={'embedding_1': 'a_embedding_1', 'embedding_2': 'a_embedding_2'}, inplace=True)

data = data.merge(embedding_df, left_on='b', right_index=True, how='left')
data.rename(columns={'embedding_1': 'b_embedding_1', 'embedding_2': 'b_embedding_2'}, inplace=True)

data = data.merge(embedding_df, left_on='c', right_index=True, how='left')
data.rename(columns={'embedding_1': 'c_embedding_1', 'embedding_2': 'c_embedding_2'}, inplace=True)

data = data.merge(embedding_df, left_on='from', right_index=True, how='left')
data.rename(columns={'embedding_1': 'from_embedding_1', 'embedding_2': 'from_embedding_2'}, inplace=True)

data = data.merge(embedding_df, left_on='to', right_index=True, how='left')
data.rename(columns={'embedding_1': 'to_embedding_1', 'embedding_2': 'to_embedding_2'}, inplace=True)

# 查看添加的新列
print(data[['a_embedding_1', 'a_embedding_2', 'b_embedding_1', 'b_embedding_2', 'c_embedding_1', 'c_embedding_2', 'from_embedding_1', 'from_embedding_2', 'to_embedding_1', 'to_embedding_2']])

        a_embedding_1  a_embedding_2  b_embedding_1  b_embedding_2  \
0            1.359785       0.666016       1.191599       1.067956   
1           -0.079549      -0.708073      -0.208072       0.480918   
2            1.652670       0.735235       0.256912       0.867778   
3            1.652670       0.735235       0.256912       0.867778   
4            1.652670       0.735235       1.191599       1.067956   
...               ...            ...            ...            ...   
282750      -0.410526       1.041433      -0.577787       0.985501   
282751      -0.410526       1.041433      -0.535084       0.517896   
282752      -0.410526       1.041433      -0.577787       0.985501   
282753      -0.390692       0.701644      -0.579520       0.793623   
282754      -0.577787       0.985501      -0.579520       0.793623   

        c_embedding_1  c_embedding_2  from_embedding_1  from_embedding_2  \
0                 NaN            NaN          1.359785          0.666016   
1      

### 统计不同城市的频率

### 对'a', 'b', 'c', 'from', 'to'进行频率编码

In [8]:
# 加载city_map
with open('../../my/encoder/city_map_频率编码.json', 'r') as f:
    city_map = json.load(f)

# 使用 city_map 替换指定列的值
columns_to_replace = ['a', 'b', 'c', 'from', 'to']

# 遍历指定列并直接用 map 映射
for col in columns_to_replace:
    data[col] = data[col].map(city_map)


print(data)

       flt_no  cap aircraft  legs  leg_no  duration  pax       a       b  \
0        7558   94      190     1       1      1.33   45    1802  119514   
1        5248  162      322     1       1      2.10  143     958   95607   
2        3524  169      738     1       1      4.52  154    5087  115275   
3        6352  167      320     1       1      4.28  167    5087  115275   
4        7516  166      320     1       1      1.33  181    5087  119514   
...       ...  ...      ...   ...     ...       ...  ...     ...     ...   
282750   7522  161      738     1       1      2.57  155   29660  168047   
282751   7676    0      32C     3       3      3.35   29   29660   21342   
282752   8304  173      73Z     1       1      2.57  166   29660  168047   
282753   7678   95      190     3       2      1.70   85   43183    4396   
282754   7677   95      190     3       2      2.25   67  168047    4396   

               c   unit_price  competitor_price  year  month  day  weekday  \
0        

### 对'flt_no', 'bd_type', 'aircraft'进行标签编码

In [9]:
data.head(5)

,flt_no,cap,aircraft,legs,leg_no,duration,pax,a,b,c,unit_price,competitor_price,year,month,day,weekday,hour,minute,from,to,a_label,b_label,c_label,from_label,to_label,a_embedding_1,a_embedding_2,b_embedding_1,b_embedding_2,c_embedding_1,c_embedding_2,from_embedding_1,from_embedding_2,to_embedding_1,to_embedding_2
0,7558,94,190,1,1,1.33,45,1802,119514,NaN,481.555556,-119.744444,2024,7,1,0,12,30,1802,119514,2,2,NaN,2,2,1.359785,0.666016,1.191599,1.067956,NaN,NaN,1.359785,0.666016,1.191599,1.067956
1,5248,162,322,1,1,2.10,143,958,95607,NaN,902.167832,0.000000,2024,7,1,0,10,30,958,95607,1,0,NaN,1,0,-0.079549,-0.708073,-0.208072,0.480918,NaN,NaN,-0.079549,-0.708073,-0.208072,0.480918
2,3524,169,738,1,1,4.52,154,5087,115275,NaN,1540.357143,283.690476,2024,7,1,0,14,15,5087,115275,2,0,NaN,2,0,1.652670,0.735235,0.256912,0.867778,NaN,NaN,1.652670,0.735235,0.256912,0.867778
3,6352,167,320,1,1,4.28,167,5087,115275,NaN,1345.449102,88.782435,2024,7,1,0,21,30,5087,115275,2,0,NaN,2,0,1.652670,0.735235,0.256912,0.867778,NaN,NaN,1.652670,0.735235,0.256912,0.867778
4,7516,166,320,1,1,1.33,181,5087,119514,NaN,593.480663,-54.049640,2024,7,1,0,23,15,5087,119514,2,2,NaN,2,2,1.652670,0.735235,1.191599,1.067956,NaN,NaN,1.652670,0.735235,1.191599,1.067956


In [10]:
import joblib
from sklearn.preprocessing import LabelEncoder
import os

# 定义需要编码的分类特征
# categorical_columns = ['flt_no', 'bd_type', 'aircraft']
categorical_columns = ['flt_no',  'aircraft']

# 从保存的文件中加载编码器并应用到data
for col in categorical_columns:
    # 加载编码器
    encoder_path = os.path.join('../../my/encoder/', f"{col}_encoder_all.pkl")
    le = joblib.load(encoder_path)
    
    try:
        # 对data进行转换
        data[col] = le.transform(data[col])
        print(f"{col}列编码完成")
    except ValueError as e:
        # 如果遇到新的类别，打印错误信息
        print(f"{col}列编码出错: {str(e)}")
        # 找出新的类别
        new_categories = set(data[col]) - set(le.classes_)
        print(f"{col}列中的新类别: {new_categories}")

# 查看编码后的结果
print("\n编码后的前几行数据：")
print(data[categorical_columns].head())

flt_no列编码完成
aircraft列编码完成

编码后的前几行数据：
   flt_no  aircraft
0    2846         0
1     622        10
2     348        46
3    1587         8
4    2784         8


## 特征和目标分离
我们要预测的是pax字段，其他字段作为特征。

In [11]:
# 特征列
# X = data[['flt_no', 'bd_type', 'cap', 'aircraft',  'leg_no', 'duration', 'a', 'b', 'c', 'year', 'month', 'day', 'weekday','holiday', 'hour', 'minute', 'second', 'from', 'to','unit_price']]
# X = data[['flt_no', 'bd_type', 'cap', 'aircraft', 'legs', 'leg_no', 'duration', 'a', 'b', 'c', 'year', 'month', 'day', 'weekday','hour', 'minute', 'second', 'from', 'to','unit_price']]
# X = data[['flt_no', 'bd_type', 'cap', 'aircraft', 'legs', 'leg_no', 'duration', 'a', 'b', 'c', 'year', 'month', 'day', 'weekday','hour', 'minute', 'second', 'from', 'to','unit_price','a_label' ,'b_label' ,'c_label' ,'from_label' ,'to_label']]

# 有abc，有标签，有嵌入
# X = data[['flt_no', 'bd_type', 'cap', 'aircraft', 'legs', 'leg_no', 'duration', 'a', 'b', 'c', 'year', 'month', 'day', 'weekday','hour', 'minute', 'second', 'from', 'to','unit_price','a_label' ,'b_label' ,'c_label' ,'from_label' ,'to_label','a_embedding_1' , 'a_embedding_2' , 'b_embedding_1','b_embedding_2' , 'c_embedding_1' , 'c_embedding_2' , 'from_embedding_1','from_embedding_2' , 'to_embedding_1' , 'to_embedding_2']]
X = data[['flt_no', 'cap', 'aircraft', 'legs', 'leg_no', 'duration', 'a', 'b', 'c', 'year', 'month', 'day', 'weekday','hour', 'minute', 'from', 'to','unit_price','competitor_price','a_label' ,'b_label' ,'c_label' ,'from_label' ,'to_label','a_embedding_1' , 'a_embedding_2' , 'b_embedding_1','b_embedding_2' , 'c_embedding_1' , 'c_embedding_2' , 'from_embedding_1','from_embedding_2' , 'to_embedding_1' , 'to_embedding_2']]
# 删除了abc，但有标签，有嵌入
# X = data[['flt_no', 'bd_type', 'cap', 'aircraft', 'legs', 'leg_no', 'duration', 'year', 'month', 'day', 'weekday','hour', 'minute', 'second', 'unit_price','a_label' ,'b_label' ,'c_label' ,'from_label' ,'to_label','a_embedding_1' , 'a_embedding_2' , 'b_embedding_1','b_embedding_2' , 'c_embedding_1' , 'c_embedding_2' , 'from_embedding_1','from_embedding_2' , 'to_embedding_1' , 'to_embedding_2']]

# 目标列
y = data['pax']

In [12]:
test_x = X.head(1)
test_x

,flt_no,cap,aircraft,legs,leg_no,duration,a,b,c,year,month,day,weekday,hour,minute,from,to,unit_price,competitor_price,a_label,b_label,c_label,from_label,to_label,a_embedding_1,a_embedding_2,b_embedding_1,b_embedding_2,c_embedding_1,c_embedding_2,from_embedding_1,from_embedding_2,to_embedding_1,to_embedding_2
0,2846,94,0,1,1,1.33,1802,119514,NaN,2024,7,1,0,12,30,1802,119514,481.555556,-119.744444,2,2,NaN,2,2,1.359785,0.666016,1.191599,1.067956,NaN,NaN,1.359785,0.666016,1.191599,1.067956


In [13]:
y.head(5)

0     45
1    143
2    154
3    167
4    181
Name: pax, dtype: int64

### 对x进行标准化

In [14]:
from sklearn.preprocessing import StandardScaler
import joblib

# 定义需要标准化的特征列（与训练时相同的特征）
features = ['flt_no', 'cap', 'aircraft', 'legs', 'leg_no', 'duration', 
           'a', 'b', 'c', 'year', 'month', 'day', 'weekday', 'hour', 'minute', 
           'from', 'to', 'unit_price','competitor_price', 'a_label', 'b_label', 'c_label', 'from_label', 'to_label',
           'a_embedding_1', 'a_embedding_2', 'b_embedding_1', 'b_embedding_2', 
           'c_embedding_1', 'c_embedding_2', 'from_embedding_1', 'from_embedding_2', 
           'to_embedding_1', 'to_embedding_2']

# 加载保存的标准化器
scaler = joblib.load('../../my/encoder/standard_scaler_x.pkl')

# 对数据进行标准化
data_scaled = scaler.transform(data[features])

# 将标准化后的数据转回DataFrame格式
data_scaled = pd.DataFrame(data_scaled, columns=features, index=data.index)

# 将标准化后的数据替换回原始数据框中
for col in features:
    data[col] = data_scaled[col]

# 查看标准化后的结果
print("标准化后的数据预览（前1行）：")
pd.set_option('display.max_columns', None)  # 显示所有列
print(data[features].head(1))

# 检查标准化后的统计信息
print("\n标准化后的统计信息：")
print(data[features].describe())

标准化后的数据预览（前1行）：
     flt_no       cap  aircraft     legs    leg_no  duration         a  \
0  0.457056 -0.813631 -1.468255 -0.84848 -0.557563 -1.200174 -1.447827   

          b   c      year     month       day   weekday      hour    minute  \
0  1.000343 NaN  1.219163  0.273793 -1.683372 -1.511763 -0.402332  0.205832   

       from        to  unit_price  competitor_price   a_label   b_label  \
0 -1.349091  0.751312   -0.534917         -0.630182  3.457104  3.961738   

   c_label  from_label  to_label  a_embedding_1  a_embedding_2  b_embedding_1  \
0      NaN    3.636126  3.634265       2.916052       0.246584       2.963021   

   b_embedding_2  c_embedding_1  c_embedding_2  from_embedding_1  \
0       1.270389            NaN            NaN          3.058969   

   from_embedding_2  to_embedding_1  to_embedding_2  
0          0.278792        2.731207        1.220578  

标准化后的统计信息：
              flt_no            cap       aircraft           legs  \
count  282755.000000  282755.000000 

### 对y进行标准化

In [15]:
# 加载y的标准化器
scaler_y = joblib.load('../../my/encoder/standard_scaler_y.pkl')

# 对目标变量进行标准化
y = data['pax']  # 假设目标列名为'pax'
# y_scaled = scaler_y.transform(y.values.reshape(-1, 1))  # 将y转换为2D数组进行标准化

# # 将标准化后的数据转回DataFrame格式
# y_scaled = pd.DataFrame(y_scaled, columns=['pax_scaled'], index=data.index)

# # 将标准化后的值添加到原始数据框中
# data['pax'] = y_scaled['pax_scaled']

## XGBoost模型预测

In [16]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np

# 加载模型
model = xgb.Booster()
model.load_model("../../my/model/频率编码/归一化_xgboost_model_1000.json")

# 转换为DMatrix格式
dmatrix = xgb.DMatrix(data_scaled)

# 进行预测
scaled_predictions = model.predict(dmatrix)

# 将标准化的预测值转换回原始尺度
predictions = scaler_y.inverse_transform(scaled_predictions.reshape(-1, 1))

In [17]:
predictions

array([[ 70.36507 ],
       [140.49496 ],
       [163.58852 ],
       ...,
       [168.31696 ],
       [ 58.76388 ],
       [ 60.971367]], dtype=float32)

In [18]:
# 正确的写法
test_results = list(zip(predictions[:100], y[:100]))  # 真实值和预测值

print("\n20条测试结果（真实值 vs 预测值）:")
for i, (true_value, pred_value) in enumerate(test_results[:40]):
    # 如果是多维数组，使用 .item() 转换为标量
    true_value = true_value.item() if isinstance(true_value, np.ndarray) else true_value
    pred_value = pred_value.item() if isinstance(pred_value, np.ndarray) else pred_value
    print(f"第{i+1}条: 真实值={true_value}, 预测值={pred_value:.2f}")



20条测试结果（真实值 vs 预测值）:
第1条: 真实值=70.36506652832031, 预测值=45.00
第2条: 真实值=140.49496459960938, 预测值=143.00
第3条: 真实值=163.58851623535156, 预测值=154.00
第4条: 真实值=164.74526977539062, 预测值=167.00
第5条: 真实值=144.90380859375, 预测值=181.00
第6条: 真实值=112.79246520996094, 预测值=138.00
第7条: 真实值=79.75504302978516, 预测值=137.00
第8条: 真实值=53.59605407714844, 预测值=50.00
第9条: 真实值=112.97456359863281, 预测值=136.00
第10条: 真实值=78.6087417602539, 预测值=99.00
第11条: 真实值=74.6685562133789, 预测值=70.00
第12条: 真实值=103.68016052246094, 预测值=62.00
第13条: 真实值=59.14217758178711, 预测值=57.00
第14条: 真实值=64.72105407714844, 预测值=67.00
第15条: 真实值=166.527587890625, 预测值=167.00
第16条: 真实值=62.43918991088867, 预测值=30.00
第17条: 真实值=103.06403350830078, 预测值=102.00
第18条: 真实值=105.48316192626953, 预测值=163.00
第19条: 真实值=92.38381958007812, 预测值=42.00
第20条: 真实值=89.43854522705078, 预测值=119.00
第21条: 真实值=67.08954620361328, 预测值=24.00
第22条: 真实值=153.0653839111328, 预测值=161.00
第23条: 真实值=177.05279541015625, 预测值=162.00
第24条: 真实值=172.7503662109375, 预测值=170.00
第25条: 真实值=178.55905151367188, 预测值

## 测试指标

In [19]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def calculate_smape(y_true, y_pred):
    """
    计算 Symmetric Mean Absolute Percentage Error (SMAPE)
    """
    y_true, y_pred = np.array(y_true).ravel(), np.array(y_pred).ravel()
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    diff = np.abs(y_pred - y_true)
    
    # 避免除以零，将分母中为零的项替换为一个小值
    denominator = np.where(denominator == 0, 1e-8, denominator)
    
    smape = 100 * np.mean(diff / denominator)
    return smape


def calculate_mape(y_true, y_pred):
    """
    计算 Mean Absolute Percentage Error (MAPE)
    """
    y_true, y_pred = np.array(y_true).ravel(), np.array(y_pred).ravel()
    
    # 避免除以零，将 y_true 中的零值替换为一个小值
    y_true = np.where(y_true == 0, 1e-8, y_true)
    
    mape = 100 * np.mean(np.abs((y_true - y_pred) / y_true))
    return mape


# 假设 y_test 和 y_pred 已经是标准化反归一化后的数据
# 将其转换为一维数组以确保形状一致
y_test = np.array(y).ravel()
y_pred = np.array(predictions).ravel()

# 评估指标
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mse)
mape = calculate_mape(y_test, y_pred)
smape = calculate_smape(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# 打印结果
print(f'Mean Squared Error (MSE): {mse:.4f}')
print(f'Root Mean Squared Error (RMSE): {rmse:.4f}')
print(f'Mean Absolute Error (MAE): {mae:.4f}')
print(f'Mean Absolute Percentage Error (MAPE): {mape:.4f}%')
print(f'Symmetric Mean Absolute Percentage Error (SMAPE): {smape:.4f}%')
print(f'R-squared (R²): {r2:.4f}')


Mean Squared Error (MSE): 497.3340
Root Mean Squared Error (RMSE): 22.3010
Mean Absolute Error (MAE): 16.8397
Mean Absolute Percentage Error (MAPE): 18.4621%
Symmetric Mean Absolute Percentage Error (SMAPE): 16.0203%
R-squared (R²): 0.8098


似乎对于较小值预测存在误差

In [20]:
def calculate_smape(y_true, y_pred):
    """
    计算 Symmetric Mean Absolute Percentage Error (SMAPE)
    """
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    smape = 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred)))
    return smape

def calculate_mape(y_true, y_pred):
    """
    计算 Mean Absolute Percentage Error (MAPE)
    """
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mape = 100 * np.mean(np.abs((y_true - y_pred) / y_true))
    return mape

In [21]:
# from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
# import numpy as np

# # 评估模型
# mse = mean_squared_error(y_test, y_pred)
# mae = mean_absolute_error(y_test, y_pred)
# rmse = np.sqrt(mse)
# mape = calculate_mape(y_test, y_pred)
# smape = calculate_smape(y_test, y_pred)

# # 打印结果
# print(f'Mean Squared Error (MSE): {mse:.4f}')
# print(f'Root Mean Squared Error (RMSE): {rmse:.4f}')
# print(f'Mean Absolute Error (MAE): {mae:.4f}')
# print(f'Mean Absolute Percentage Error (MAPE): {mape:.4f}%')
# print(f'Symmetric Mean Absolute Percentage Error (SMAPE): {smape:.4f}%')

## 保存模型

## 超参数设置

## 不同特征重要程度测试

In [22]:
# import xgboost as xgb
# import matplotlib.pyplot as plt

# # 假设 model 是训练好的 XGBoost 模型
# xgb.plot_importance(model, importance_type='weight', title="Feature Importance (Weight)", height=0.5)
# plt.show()

# xgb.plot_importance(model, importance_type='gain', title="Feature Importance (Gain)", height=0.5)
# plt.show()

# xgb.plot_importance(model, importance_type='cover', title="Feature Importance (Cover)", height=0.5)
# plt.show()